## Data Cleaning for Ag slab w/ [C, H, O] adsorbates
#### DropBox --> filtered folder names

In [1]:
import os
import time
from dotenv import load_dotenv
import dropbox
from dropbox.files import SharedLink
from pymatgen.io.vasp.inputs import Incar

load_dotenv(override=True)
TOKEN = os.getenv("DROPBOX_TOKEN")

In [5]:
dbx = dropbox.Dropbox(TOKEN)

ROOT_PATH = SharedLink(url="https://www.dropbox.com/scl/fo/qpg1zuo3g7vb3il1wmqy3/AA4wERzz28lJhYTCAcSEvqk?dl=0")

results = []
def list_folder():
    res = dbx.files_list_folder(path="", shared_link=ROOT_PATH)
    
    while True:
        for entry in res.entries:
            results.append({
                "name": entry.name,
                "type": entry.__class__.__name__
            })
        if not res.has_more:
            break
        res = dbx.files_list_folder_continue(res.cursor)

list_folder()
print(f"Total items: {len(results)}")

# write txt file of folder names
with open("cache/dropbox_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in results:
        if entry["type"] == "FolderMetadata":
            f.write(entry["name"] + "\n")

print("Saved to cache/dropbox_foldernames.txt")

Total items: 1079
Saved to cache/dropbox_foldernames.txt


In [6]:
# read in raw folder names list from cache
with open("cache/dropbox_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

# filter out obviously bad data via substring matching
exclude = {
    "wrong",
    "toomanykpoints",
    "oldbad",
    "exploded",
        }

foldernames = [n for n in foldernames if not any(term in n for term in exclude)]
print(f"Total items: {len(foldernames)}")

Total items: 1031


In [8]:
# filter out elements and calculation types we don't want
exclude = {
    "Al", "Au", "Co", "Cr", "Cu", "Fe", "Ga", "In", "Ir", "Mg", "Mn", "Mo", "Ni", "Pd", "Pt", "Re", "Rh", "Ru", "Sc", "Sn", "Ti", "V", "W", "Zn"
    "minhop", "CL", "CL_JS", "Bader", "Bare", "bare", "Bulk", "Gas", "gas",
}

foldernames = [n for n in foldernames if not any(term in n for term in exclude)]
print(f"Total items: {len(foldernames)}")

Total items: 282


In [9]:
# check if OUTCAR exists and if so, get elements from POSCAR

elements_map = {}
irregular_poscars = {}
no_outcar = []
failed_folders = {}

for folder in foldernames:
    folder_path = f"/{folder}"
    try:
        res = dbx.files_list_folder(path=folder_path, shared_link=ROOT_PATH)
        names = {e.name for e in res.entries}
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)
        continue

    if "OUTCAR" not in names:
        no_outcar.append(folder)
        continue

    try:
        _, dres = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"{folder_path}/POSCAR")
        lines = dres.content.decode("utf-8", errors="ignore").splitlines()
        tokens = lines[5].split()
        if all(t.isalpha() for t in tokens):
            elements_map[folder] = set(tokens)
        else:
            irregular_poscars[folder] = lines[:5]
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)

    time.sleep(0.2)

In [12]:
print(f"""
    elements_map: {len(elements_map)} items
    irregular_poscars: {len(irregular_poscars)} items,
    no_outcar": {len(no_outcar)} items,
    failed_folders: {len(failed_folders)} items,
    """)


    elements_map: 227 items
    irregular_poscars: 15 items,
    no_outcar": 38 items,
    failed_folders: 2 items,
    


In [13]:
# inspect irregular POSCARs

for folder, lines in irregular_poscars.items():
    print(folder)
    print("\n".join(lines))
    print("-"*10)

1OAg
 O Ag 
 1.0000000000000000
     8.6296027574400007    0.0000000000000000    0.0000000000000000
     4.3148013787200004    7.4734547894199999    0.0000000000000000
     0.0000000000000000    0.0000000000000000   23.4868010454999983
----------
O2TSZn
 O Ag Zn Ag 
 1.0000000000000000
     8.6296027574400007    0.0000000000000000    0.0000000000000000
     4.3148013787200004    7.4734547894199999    0.0000000000000000
     0.0000000000000000    0.0000000000000000   23.4868010454999983
----------
LayerCalcAg0
Ag 
 1.0000000000000000
     8.6209398102092099    0.0000000000000000    0.0000000000000000
     4.3104699051046049    7.4659528801377721    0.0000000000000000
     0.0000000000000000    0.0000000000000000   31.7316131323659008
----------
O6Ag_OnAg
Ag  O 
 1.0000000000000000
     8.6296027574400007    0.0000000000000000    0.0000000000000000
     4.3148013787200004    7.4734547894199999    0.0000000000000000
     0.0000000000000000    0.0000000000000000   23.4868010454999983
-----

In [14]:
allowed = {"Ag", "C", "H", "O"}
clean_map = {k: v for k, v in elements_map.items() if v.issubset(allowed)}

print(f"Total items: {len(clean_map)}")

Total items: 186


In [15]:
# last round of misc. filtering from manual inspection

exclude = {"TS", "spin"}
clean_map = {
    k: v for k, v in clean_map.items()
    if v.issubset(allowed) and not any(sub in k for sub in exclude)
}

# delete folder if folder_good exists
suffixed = {k[:-5] for k in clean_map if k.endswith("_good")}
clean_map = {k: v for k, v in clean_map.items() if k not in suffixed}

print(f"Total items: {len(clean_map)}")

Total items: 163


In [16]:
# write txt file of cleaned folder names
with open("cache/cleaned_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in clean_map:
        f.write(entry + "\n")

print("Saved to cache/cleaned_foldernames.txt")

Saved to cache/cleaned_foldernames.txt


In [17]:
# get INCARs for clean folders

incar_map = {}
missing_incars = []

for folder in clean_map:
    try:
        _, res = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"/{folder}/INCAR")
        incar_map[folder] = Incar.from_str(res.content.decode("utf-8", errors="ignore"))
    except dropbox.exceptions.ApiError:
        missing_incars.append(folder)
    time.sleep(0.2)

print(f"Total items: f{len(incar_map)}, Missing INCARs: {len(missing_incars)}")

Total items: f163, Missing INCARs: 0


In [18]:
# filter out DFT calculations other than relaxations

filtered_incar_map = {
    k: v for k, v in incar_map.items()
    if v.get("IBRION") in (1, 2, 3) and v.get("ISPIN", 1) == 1 and v.get("NSW", 0) != 0
}
print(f"Total items: {len(filtered_incar_map)}")

Total items: 143


In [19]:
# write txt file of cleaned and selected folder names (relaxations)
with open("cache/selected_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in filtered_incar_map:
        f.write(entry + "\n")

print("Saved to cache/selected_foldernames.txt")

Saved to cache/selected_foldernames.txt


continue data preparation in `ag-data-selection.ipynb`

In [ ]:
# if needed for another round of filtering: read in raw folder names list from cache

with open("cache/selected_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

foldernames = [v for v in foldernames if not any(sub in v for sub in exclude)]

print(f"Total items: {len(foldernames)}")

Total items: 143
